# semiotic-emergence-gpu

GPU-accelerated evolutionary communication simulation.

**Runtime setup:** Go to Runtime > Change runtime type > select **T4 GPU** before running.

In [ ]:
# 1. Install
!git clone https://github.com/onblueroses/semiotic-emergence-gpu.git
%cd semiotic-emergence-gpu
!pip install -q -e '.[dev]'
!pip install -q --upgrade 'jax[cuda12-local]'

In [ ]:
# 2. Verify GPU
import jax
print(f"Backend: {jax.default_backend()}")
print(f"Devices: {jax.devices()}")
assert jax.default_backend() == "gpu", "No GPU detected! Change runtime type."

In [ ]:
# 3. Run tests
!python -m pytest tests/ -q

In [ ]:
# 4. Baseline: 384 pop, 56 grid, 500 ticks, 50 gens
import os, time
os.makedirs("runs/baseline", exist_ok=True)
os.chdir("runs/baseline")

start = time.time()
!python -m semgpu.main 42 50 --pop 384 --grid 56 --ticks 500
elapsed = time.time() - start
print(f"\nTotal: {elapsed:.1f}s ({elapsed/50:.2f}s/gen)")

os.chdir("/content/semiotic-emergence-gpu")

In [ ]:
# 5. VRAM after baseline
!nvidia-smi --query-gpu=name,memory.used,memory.total,utilization.gpu --format=csv,noheader

In [ ]:
# 6. Scale test: 5k prey (previously OOM'd before spatial binning)
os.makedirs("runs/scale-5k", exist_ok=True)
os.chdir("runs/scale-5k")

start = time.time()
!python -m semgpu.main 0 10 --pop 5000 --grid 150 --food 1000 --ticks 100
elapsed = time.time() - start
print(f"\nTotal: {elapsed:.1f}s ({elapsed/10:.2f}s/gen)")

os.chdir("/content/semiotic-emergence-gpu")

In [ ]:
# 7. VRAM after 5k
!nvidia-smi --query-gpu=name,memory.used,memory.total,utilization.gpu --format=csv,noheader

In [ ]:
# 8. Scale test: 10k prey
os.makedirs("runs/scale-10k", exist_ok=True)
os.chdir("runs/scale-10k")

start = time.time()
!python -m semgpu.main 0 5 --pop 10000 --grid 200 --food 2000 --ticks 100
elapsed = time.time() - start
print(f"\nTotal: {elapsed:.1f}s ({elapsed/5:.2f}s/gen)")

os.chdir("/content/semiotic-emergence-gpu")

In [ ]:
# 9. VRAM after 10k
!nvidia-smi --query-gpu=name,memory.used,memory.total,utilization.gpu --format=csv,noheader

In [ ]:
# 10. Scale test: 50k prey (pushing T4 limits - may OOM)
os.makedirs("runs/scale-50k", exist_ok=True)
os.chdir("runs/scale-50k")

start = time.time()
try:
    !python -m semgpu.main 0 3 --pop 50000 --grid 450 --food 5000 --ticks 50
    elapsed = time.time() - start
    print(f"\nTotal: {elapsed:.1f}s ({elapsed/3:.2f}s/gen)")
except Exception as e:
    print(f"Failed: {e}")

os.chdir("/content/semiotic-emergence-gpu")

In [ ]:
# 11. VRAM after 50k
!nvidia-smi --query-gpu=name,memory.used,memory.total,utilization.gpu --format=csv,noheader

In [ ]:
# 12. Inspect results from largest successful run
import pandas as pd

# Find the largest scale run that completed
for name in ['scale-50k', 'scale-10k', 'scale-5k', 'baseline']:
    path = f"runs/{name}/output.csv"
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f"=== {name} ({len(df)} gens) ===")
        print(df[["generation", "avg_fitness", "max_fitness", "signals_emitted",
                  "mutual_info", "iconicity", "zone_deaths", "signal_entropy"]].tail())
        print()

In [ ]:
# 13. Download all results
!ls -la runs/*/output.csv 2>/dev/null

# Uncomment to download:
# from google.colab import files
# !tar czf results.tar.gz runs/
# files.download('results.tar.gz')